<div align="right"><a href="L08-Notch-Filter.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). Please note that for running MATLAB in Notebooks, you have to install the [MATLAB API for Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) first. The installation process is described [here](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/S08-Notch-Filter.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S07 Convolution Matrix](./S07-Convolution-Matrix.ipynb) | [Exercise 08](./E08-Notch-Filter.ipynb) | [Contents / Overview](./00-MATLAB-Notebooks-Overview-EN.ipynb) | Next: [S09 Sampling and Reconstruction](./S09-Sampling-and-Reconstruction.ipynb)

</div>

---


## Exercise: Notch Filter (2.X)

This exercise should be a design example of a recursive notch filter.  
Its effect is illustrated by some examples.

A 2nd order notch-filter is designed in the Z-domain.  
It has conjugate complex zeros on the unit circle — corresponding to frequency $\Omega_0$ that should be suppressed — and poles at the same phase angles that are marginally inside the unit circle.

**Zeros:** $e^{\pm j \Omega_0}$  
**Poles:** $\beta e^{\pm j \Omega_0}$

### a)
Write a MATLAB function `notchdesign()` to calculate the filter coefficients  
$\mathbf{a} = \{a_0, a_1, a_2\}$ and $\mathbf{b} = \{b_0, b_1, b_2\}$.  
This function should use the normalized frequency $\Omega_0$ and $\beta$ as parameters.  
Calculate the coefficients either by analytical calculations or by using function `poly()`.

### b)
Plot the frequency response (in dB) of a notch filter ($\Omega_N = \pi/4$, $\beta = 0.9$) using function `freqz()`.

What is the influence on the frequency response if you increase factor $\beta$ to $0.9999$?  
Verify your guess using MATLAB.

### c)
This exercise demonstrates the effects of a notch filter by an audio example.

Create a test signal $x(k) = x_1(k) + x_2(k)$ that consists of the following components:

| Component | Description |
|------------|--------------|
| **1. sinus component** $x_1(k)$ | |
| frequency | $f_{sin} = 1000 \text{ Hz}$ |
| amplitude | $-0.5 \dots 0.5$ |
| **2. noise component** $x_2(k)$ | |
| distribution | equally distributed between $-0.25$ and $0.25$ |

Use a sampling frequency of $f_a = 8000 \text{ Hz}$.  
The test signal should span an interval of $5\,\mathrm{s}$.

Output the test signal $x(k)$ within MATLAB using function `sound()` to a loudspeaker.  
Use a notch filter ($\beta = 0.9$) to suppress the sinusoidal signal $x_1(k)$ of $x(k)$.  
Only a noise signal should remain.

For filtering you may use MATLAB’s function `filter()`.  
Output the filtered signal using function `sound()` to hear the effect of the notch filter.

### d)
Repeat the audio example of exercise (c).  
Use a notch filter with $\beta = 0.9999$.  
What has changed compared to the case of $\beta = 0.9$?  
What is the reason for this change?

Check the observed behavior by plotting the impulse responses of the notch filters with $\beta = 0.9$ and $\beta = 0.9999$.  
(Function `[h,t] = impz(b,a,N,fa);` may be useful.)

### e) Additional exercise — practical example
If you receive some programs using your shortwave radio, you may notice distortions by sinusoidal signals.  
This might be carriers of other shortwave stations.

To demonstrate this behavior, there is the file  
`l936.28_kHz_25.04.2001_23_17.wav`.  
The distortion signal was measured at about $f = 345 \text{ Hz}$.

Now it’s your task to eliminate the distortion with some lines of MATLAB code and output the cleaned signal through a loudspeaker.

**Hints:**
- Use MATLAB’s functions `wavread()` or `wavwrite()` to handle the wave file.
- To fix $\beta$, select a value according to your experiences of exercises (b) and (d).
- Scale the filtered signal to have a maximum amplitude of 1.
- MATLAB’s function `sound()` doesn’t work on any operating system with a sample rate of $44.1 \text{ kHz}$.  
  Use a separate program to play the output file (e.g., `sdtaudio` with Solaris 8).

In [ ]:
# BEGIN SOLUTION
% #####  Part b:  Dimensioning and Frequency Response  #####
Omega0 = pi/4;

for beta = [0.9 0.9999]
    [a,b] = snotchdesign(Omega0,beta);
    [H,Omega] = freqz(b,a);

    figure;
    plot(Omega/pi,20*log10(abs(H)));
    title(['Frequency Response \beta=',num2str(beta)]);
    xlabel('\Omega / \pi');
    ylabel('|H(e^{j\Omega})| in dB');
    ylim([-60 10]);
    grid on;
end;

disp('Continuing with Part c): Press any key...'); pause;

% #####  Part c:  Audio Demonstration  #####
fa = 8000;     % Sampling frequency
fsinus = 1000; % Frequency of the sinusoidal signal
duration = 5;  % Duration of the signal in seconds
N = fa * duration;  % Number of samples

% Generate the noise+sinusoidal signal
x1 = 0.5 * sin(2*pi*fsinus/fa * [1:N]');
x2 = (rand(N,1) - 0.5) / 2;
x = x1 + x2;

disp('Playing: x(k) unfiltered'); 
sound(x, fa);
pause(length(x) / fa); 


beta = 0.9; 
Omega0 = 2*pi*fsinus/fa;      
[a, b] = snotchdesign(Omega0, beta); 
y = filter(b, a, x);

disp(['Playing: y(k) filtered beta=', num2str(beta)]); 
sound(y, fa);
pause(length(y) / fa); 


disp('Continuing with Part d): Press any key...'); 
pause;

% #####  Part d:  Audio Demonstration, Length of the Impulse Response  #####
disp('Playing: x(k) unfiltered'); 
sound(x, fa);
pause(length(x) / fa); 


beta = 0.9999;
[a, b] = snotchdesign(Omega0, beta); 
y = filter(b, a, x);

disp(['Playing: x(k) filtered beta=', num2str(beta)]); 
sound(y, fa);
pause(length(y) / fa); 


for beta = [0.9 0.9999]
    [a, b] = snotchdesign(Omega0, beta);

    [h, t] = impz(b, a, N, fa);

    % Use fewer data points for plotting
    t = t(1:10:length(h)); 
    h = h(1:10:length(h));

    % # graphical output # %
    figure;
    stem(t, h);
    axis([0 duration -1e-4 1e-4]);
    title(['Impulse Response \beta=', num2str(beta)]); 
    xlabel('t=k T_A in s'); 
    ylabel('h(k)');
end;

% #####  Part e:  Radio Signal  #####
[x, fa] = audioread('l936.28_kHz_25.04.2001_23_17.wav');  

f0 = 345;
beta = 0.9; 
Omega0 = 2*pi*f0/fa;      
[a, b] = snotchdesign(Omega0, beta); 
y = filter(b, a, x);
y = y / max(abs(y)); % Scale for output
audiowrite('l936.28_kHz_25.04.2001_23_17.result.wav', y, fa);
# END SOLUTION

## Solution: Notch Filter (2.X)

### a)
Zeros: $e^{\pm j \Omega_N}$  
Poles: $\beta e^{\pm j \Omega_N}$

The transfer function $H(z)$ and therefore the filter coefficients are obtained through multiplication:

$$
\begin{array}{rcl}
H(z) & = & \frac{\left( z -  e^{+ j \Omega_N} \right) \left( z -  e^{- j \Omega_N} \right)}
{ \left( z - \beta e^{+ j \Omega_N} \right) \left( z - \beta e^{- j \Omega_N} \right)} \\
     & = &\frac{z^2 - z \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + 1}
{z^2 - z \beta \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + \beta^2} \\
     & = &\frac{z^2 - 2 z \cos\left( \Omega_N \right) + 1}
{z^2 - 2 \beta z \cos\left( \Omega_N \right) + \beta^2} \\
     & = &\frac{1 - 2 z^{-1} \cos\left( \Omega_N \right) + z^{-2}}
{1 - 2 \beta z^{-1} \cos\left( \Omega_N \right) + \beta^2 z^{-2}}
\end{array}
$$

With the normal definition of the coefficient pair $\mathbf{b}$ and $\mathbf{a}$

$$
H(z) = \frac{\sum\limits_{\mu=0}^{2} b_\mu z^{-\mu}}{1 + \sum\limits_{\nu=1}^{2} a_\nu z^{-\nu}}
$$

the values are:

$$
\mathbf{b} = \{1, -2\cos(\Omega_N), 1\}
$$

$$
\mathbf{a} = \{1, -2\beta\cos(\Omega_N), \beta^2\}
$$

### b)
It is clearly shown that the notch in the frequency domain becomes narrower if the poles (through factor $\beta$) are closer to the unit circle.

<img src="./pngs/ldsv2a19_b1.png" width="250">
<img src="./pngs/ldsv2a19_b2.png" width="250">

### c)
...listen and enjoy...

### d)
We hear that the notch filter with $\beta = 0.9999$ starts to affect the signal very slowly.  
This is because of the extremely long settling time of the filter.

The transient phenomenon is also reflected in the impulse response.

<img src="./pngs/ldsv2a19_d1.png" width="250">
<img src="./pngs/ldsv2a19_d2.png" width="250">

Explanation: Because the signal in the frequency domain is very narrow, the signal must be longer in the time domain.  
(Uncertainty principle of communications)

### e)
With appropriately chosen coefficients, the disturbance can be nearly eliminated.